<a href="https://colab.research.google.com/github/mariangeldante2563/ProyectoCOLAPBoomil/blob/main/Copia_de_BOOMIL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## CELDA 1 — Instalación completa

In [ ]:
!pip install nest_asyncio websockets pandas numpy scikit-learn pyarrow statsmodels -q


## CELDA 2 — Configuración segura + Google Drive

**Importante:** El token API se carga desde Colab Secrets (ícono llave en el panel izquierdo).
Nunca se guarda en el código.

In [ ]:
import nest_asyncio, asyncio, json, os, logging, time
from datetime import datetime, timezone
import websockets
import pandas as pd
import numpy as np

nest_asyncio.apply()

# Montar Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# Token seguro desde Colab Secrets
try:
    from google.colab import userdata
    _TOKEN = userdata.get("DERIV_TOKEN")
    if not _TOKEN:
        raise ValueError
    print("Token cargado desde Colab Secrets.")
except Exception:
    _TOKEN = os.environ.get("DERIV_TOKEN", "")
    if not _TOKEN:
        raise RuntimeError("Configura DERIV_TOKEN en Colab Secrets (icono llave en panel izquierdo)")

CFG = {
    "APP_ID"        : 127820,
    "TOKEN"         : _TOKEN,
    "SYMBOL"        : "BOOM1000",
    "SAVE_DIR"      : "/content/drive/MyDrive/BOOMIL_DATA",
    "CHECKPOINT"    : "/content/drive/MyDrive/BOOMIL_DATA/checkpoint.json",
    "MASTER_FILE"   : "/content/drive/MyDrive/BOOMIL_DATA/master.parquet",
    "BATCH_SIZE"    : 300,
    "TARGET_TICKS"  : 90_000,
    "SESSION_LIMIT" : 41_400,   # 11.5h x 1 tick/s x 3600s/h = margen seguro antes del corte de Colab
    "TICK_TIMEOUT"  : 15,
    "WS_PING_INT"   : 20,
    "MAX_RECONNECTS": 999_999,
    "BASE_WAIT"     : 3,
    "MAX_WAIT"      : 120,
}

os.makedirs(CFG["SAVE_DIR"], exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger("BOOMIL")

def check_session_state():
    cp_path = CFG["CHECKPOINT"]
    if not os.path.exists(cp_path):
        print("SESION 1 — Primera ejecución. No hay checkpoint previo.")
        return None
    try:
        with open(cp_path) as f:
            cp = json.load(f)
        total  = cp.get("total_ticks", 0)
        target = CFG["TARGET_TICKS"]
        pct    = total / target * 100
        remain = target - total
        print(f"Reanudando desde checkpoint.")
        print(f"Ticks guardados : {total:,}  ({pct:.1f}%)")
        print(f"Ticks restantes : {remain:,}")
        if remain > CFG["SESSION_LIMIT"]:
            sessions_left = int(np.ceil(remain / CFG["SESSION_LIMIT"]))
            print(f"Sesiones restantes de Colab necesarias: {sessions_left}")
        else:
            print("Esta sesion es suficiente para terminar.")
        return cp
    except Exception as e:
        print(f"Error leyendo checkpoint: {e}")
        return None

state = check_session_state()
log.info("Config lista. Dir: %s", CFG["SAVE_DIR"])


## CELDA 3 — Colector resiliente con SESSION_LIMIT, reconexión automática y flush a Drive

In [ ]:
def cp_load() -> dict:
    if os.path.exists(CFG["CHECKPOINT"]):
        try:
            with open(CFG["CHECKPOINT"]) as f:
                return json.load(f)
        except Exception:
            log.warning("Checkpoint corrupto. Reiniciando.")
    return {"total_ticks": 0, "files": [], "last_epoch": None, "sessions": 0}

def cp_save(cp: dict) -> None:
    tmp = CFG["CHECKPOINT"] + ".tmp"
    with open(tmp, "w") as f:
        json.dump(cp, f, indent=2)
    os.replace(tmp, CFG["CHECKPOINT"])

def flush_batch(batch: list, cp: dict) -> None:
    if not batch:
        return
    df_b = pd.DataFrame(batch)
    df_b["datetime"] = pd.to_datetime(df_b["epoch"], unit="s", utc=True)
    ts  = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    idx = len(cp["files"]) + 1
    path = os.path.join(CFG["SAVE_DIR"], f"ticks_{CFG['SYMBOL']}_{ts}_{idx:05d}.parquet")
    df_b.to_parquet(path, index=False, compression="snappy")
    cp["files"].append(path)
    cp["total_ticks"] += len(batch)
    cp["last_epoch"]   = int(df_b["epoch"].iloc[-1])
    cp_save(cp)
    log.info("Flush #%d | +%d ticks | Total: %s/%s (%.1f%%)",
             idx, len(batch), f"{cp['total_ticks']:,}", f"{CFG['TARGET_TICKS']:,}",
             cp["total_ticks"] / CFG["TARGET_TICKS"] * 100)

def progress_bar(current, total, width=40):
    filled = int(width * current / max(total, 1))
    bar    = "#" * filled + "-" * (width - filled)
    pct    = current / max(total, 1) * 100
    return f"[{bar}] {pct:.1f}% ({current:,}/{total:,})"

async def _ws_session(cp: dict, target: int, session_budget: int) -> str:
    url = f"wss://ws.derivws.com/websockets/v3?app_id={CFG['APP_ID']}"
    async with websockets.connect(
        url,
        ping_interval=CFG["WS_PING_INT"],
        ping_timeout=10,
        close_timeout=10,
        max_size=2**20,
    ) as ws:
        await ws.send(json.dumps({"authorize": CFG["TOKEN"]}))
        auth = json.loads(await asyncio.wait_for(ws.recv(), timeout=15))
        if auth.get("error"):
            raise ConnectionError(f"Auth: {auth['error']['message']}")
        log.info("Autenticado | loginid=%s", auth["authorize"]["loginid"])
        await ws.send(json.dumps({"ticks": CFG["SYMBOL"], "subscribe": 1}))
        log.info("Suscrito a %s", CFG["SYMBOL"])

        batch         = []
        session_ticks = 0
        t_start       = time.time()

        while True:
            if cp["total_ticks"] + len(batch) >= target:
                flush_batch(batch, cp)
                return "done"
            if session_ticks >= session_budget:
                flush_batch(batch, cp)
                log.warning("Limite de sesion (%d ticks) alcanzado. Abre nueva sesion de Colab.", session_budget)
                return "session_limit"

            raw = await asyncio.wait_for(ws.recv(), timeout=CFG["TICK_TIMEOUT"])
            msg = json.loads(raw)

            if msg.get("error"):
                raise ConnectionError(f"API error: {msg['error']['message']}")

            if msg.get("msg_type") == "tick":
                t = msg["tick"]
                batch.append({
                    "symbol": t["symbol"],
                    "epoch" : int(t["epoch"]),
                    "quote" : float(t["quote"]),
                })
                session_ticks += 1

                if len(batch) >= CFG["BATCH_SIZE"]:
                    flush_batch(batch, cp)
                    batch = []

                if session_ticks % 1_800 == 0:
                    total_now = cp["total_ticks"]
                    elapsed   = (time.time() - t_start) / 3600
                    rate      = session_ticks / max(elapsed * 3600, 1)
                    remain_s  = (CFG["SESSION_LIMIT"] - session_ticks) / max(rate, 1)
                    print(f"\n{progress_bar(total_now, target)}")
                    print(f"Sesion: {session_ticks:,} ticks | Transcurrido: {elapsed:.2f}h | Budget restante: {remain_s/3600:.2f}h\n")

async def collect(target: int = None, session_limit: int = None) -> dict:
    target        = target        or CFG["TARGET_TICKS"]
    session_limit = session_limit or CFG["SESSION_LIMIT"]
    cp            = cp_load()
    cp["sessions"] = cp.get("sessions", 0) + 1
    cp_save(cp)

    log.info("SESION %d | Objetivo: %s | Guardados: %s | Faltan: %s | Budget sesion: %s",
             cp["sessions"], f"{target:,}", f"{cp['total_ticks']:,}",
             f"{target - cp['total_ticks']:,}", f"{session_limit:,}")

    if cp["total_ticks"] >= target:
        log.info("Objetivo ya alcanzado en sesiones anteriores.")
        return cp

    print(progress_bar(cp["total_ticks"], target))
    print()

    reconnects = 0
    while cp["total_ticks"] < target and reconnects < CFG["MAX_RECONNECTS"]:
        try:
            result = await _ws_session(cp, target, session_limit)
            if result == "done":
                log.info("OBJETIVO ALCANZADO! %d ticks en %d sesion(es).", cp["total_ticks"], cp["sessions"])
                break
            elif result == "session_limit":
                log.warning("FIN DE SESION COLAB\nDatos guardados: %s ticks\nCheckpoint: %s\nEjecuta las celdas 1,2,3 en nueva sesion.",
                            f"{cp['total_ticks']:,}", CFG["CHECKPOINT"])
                break
            reconnects = 0

        except asyncio.TimeoutError:
            reconnects += 1
            wait = min(CFG["BASE_WAIT"] * (2**min(reconnects, 6)), CFG["MAX_WAIT"])
            log.warning("Timeout | intento #%d | %ds...", reconnects, wait)
            await asyncio.sleep(wait)

        except (websockets.exceptions.ConnectionClosed, ConnectionError, OSError) as exc:
            reconnects += 1
            wait = min(CFG["BASE_WAIT"] * (2**min(reconnects, 6)), CFG["MAX_WAIT"])
            log.warning("Desconexion (%s) | intento #%d | %ds...", exc, reconnects, wait)
            await asyncio.sleep(wait)

        except Exception as exc:
            reconnects += 1
            wait = min(CFG["BASE_WAIT"] * (2**min(reconnects, 6)), CFG["MAX_WAIT"])
            log.error("Error: %s | intento #%d | %ds...", exc, reconnects, wait)
            await asyncio.sleep(wait)

    log.info("Estado final | %s ticks | %d archivos | %d sesion(es)",
             f"{cp['total_ticks']:,}", len(cp["files"]), cp["sessions"])
    return cp

cp_final = asyncio.run(collect())


## CELDA 4 — Verificación de checkpoint

Ejecutar al inicio de cada sesión nueva para ver el progreso guardado.

In [ ]:
import json, os
import pandas as pd

CHECKPOINT = CFG["CHECKPOINT"]
TARGET     = CFG["TARGET_TICKS"]
SESSION_CAP = CFG["SESSION_LIMIT"]

if not os.path.exists(CHECKPOINT):
    print("Sin datos previos. Esta es tu Sesion 1.")
    print(f"Necesitas ~{TARGET/SESSION_CAP:.1f} sesiones de Colab.")
else:
    with open(CHECKPOINT) as f:
        cp = json.load(f)

    total     = cp["total_ticks"]
    sessions  = cp.get("sessions", "?")
    files     = len(cp.get("files", []))
    remaining = max(TARGET - total, 0)
    pct       = total / TARGET * 100

    print(f"Progreso  : {total:>10,} / {TARGET:,} ({pct:.1f}%)")
    print(f"Sesiones  : {sessions}")
    print(f"Archivos  : {files}")
    print(f"Restantes : {remaining:>10,} ticks")

    if remaining <= 0:
        print("OBJETIVO COMPLETADO. Puedes proceder al analisis.")
    else:
        sess_left = int(-(-remaining // SESSION_CAP))
        print(f"Sesiones Colab restantes: {sess_left}")
        print("Ejecuta celdas 1, 2 y 3 para continuar.")

    missing = [f for f in cp.get("files", []) if not os.path.exists(f)]
    if missing:
        print(f"ADVERTENCIA: {len(missing)} archivos no encontrados en Drive")
    else:
        print(f"Todos los {files} archivos Parquet estan intactos.")


## CELDA 5 — Consolidar todos los parquet en master DataFrame

In [ ]:
import glob

parquet_files = sorted(glob.glob(os.path.join(CFG["SAVE_DIR"], "ticks_*.parquet")))
print(f"Archivos encontrados: {len(parquet_files)}")

if len(parquet_files) == 0:
    raise FileNotFoundError("No hay archivos parquet. Ejecuta primero la coleccion de ticks.")

dfs = []
for f in parquet_files:
    try:
        dfs.append(pd.read_parquet(f))
    except Exception as e:
        print(f"Error leyendo {f}: {e}")

df_master = pd.concat(dfs, ignore_index=True)

# Deduplicar por epoch (evita duplicados si se ejecuta varias veces)
df_master = df_master.drop_duplicates(subset=["epoch"]).sort_values("epoch").reset_index(drop=True)

# Index temporal con UTC explícito (evita ambigüedad de zona horaria)
df_master["datetime"] = pd.to_datetime(df_master["epoch"], unit="s", utc=True)
df_master = df_master.set_index("datetime")

print(f"Total ticks unicos   : {len(df_master):,}")
print(f"Rango temporal       : {df_master.index[0]} -> {df_master.index[-1]}")
print(f"Duracion total       : {(df_master.index[-1] - df_master.index[0]).total_seconds()/3600:.1f} horas")
df_master.head()


## CELDA 6 — Feature engineering SIN look-ahead bias

In [ ]:
import numpy as np

df = df_master.copy()

# 1) Retorno logarítmico por tick
df["ret_log"] = np.log(df["quote"]).diff()

# 2) Volatilidad rolling
df["vol_20"] = df["ret_log"].rolling(20).std()
df["vol_50"] = df["ret_log"].rolling(50).std()

# 3) Medias móviles
df["ma_20"] = df["quote"].rolling(20).mean()
df["ma_50"] = df["quote"].rolling(50).mean()
df["ma_200"] = df["quote"].rolling(200).mean()

# 4) Señal de cruce de medias
df["signal_ma"] = np.where(df["ma_20"] > df["ma_50"], 1, -1)

# 5) RSI (14 periodos)
delta = df["quote"].diff()
gain  = delta.clip(lower=0).rolling(14).mean()
loss  = (-delta.clip(upper=0)).rolling(14).mean()
rs    = gain / loss.replace(0, np.nan)
df["rsi_14"] = 100 - (100 / (1 + rs))

# 6) Sesión intradía (hora UTC)
hour = df.index.hour
df["session"] = np.select(
    [(hour >= 0) & (hour < 8),
     (hour >= 8) & (hour < 16),
     (hour >= 16) & (hour < 24)],
    ["asia", "europa", "america"],
    default="otro"
)

# 7) Detección de spikes SIN look-ahead bias
# Usamos rolling quantile sobre ventana pasada de 1000 ticks (sin datos futuros)
df["diff_quote"] = df["quote"].diff()
df["spike_thresh_rolling"] = df["diff_quote"].rolling(1000, min_periods=100).quantile(0.995)
df["is_spike"] = (df["diff_quote"] >= df["spike_thresh_rolling"]).astype(int)

# 8) Ticks desde el último spike (corrección del bug de cumsum/cumcount)
# Método correcto: acumular ticks desde el último spike
spike_positions = df.index[df["is_spike"] == 1]
df["ticks_since_spike"] = len(df)  # valor por defecto alto
for i, idx in enumerate(spike_positions):
    next_spike = spike_positions[i+1] if i+1 < len(spike_positions) else df.index[-1]
    mask = (df.index >= idx) & (df.index <= next_spike)
    df.loc[mask, "ticks_since_spike"] = range(sum(mask))

# 9) Bollinger Bands
df["bb_mid"]   = df["quote"].rolling(20).mean()
df["bb_std"]   = df["quote"].rolling(20).std()
df["bb_upper"] = df["bb_mid"] + 2 * df["bb_std"]
df["bb_lower"] = df["bb_mid"] - 2 * df["bb_std"]
df["bb_pos"]   = (df["quote"] - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"] + 1e-10)

print(f"Spikes detectados: {df['is_spike'].sum()}")
print(f"Tasa de spikes   : {df['is_spike'].mean()*100:.3f}%")
df.tail()


## CELDA 7 — Target ML correcto

Se corrige el orden de `rolling` y `shift` para evitar look-ahead bias en el target.

In [ ]:
# Horizonte de predicción: próximos 10 ticks
horizon = 10

# Spike futuro correcto: ventana FUTURA de horizon ticks
# Corrección: rolling primero, luego shift (orden correcto)
df["future_spike"] = (
    df["is_spike"]
    .rolling(horizon)
    .max()
    .shift(-horizon)
    .fillna(0)
    .astype(int)
)

print("Distribución del target:")
print(df["future_spike"].value_counts())
print(f"Tasa de spikes futuros: {df['future_spike'].mean()*100:.3f}%")
df[["is_spike", "future_spike"]].tail(20)


## CELDA 8 — Preparar dataset ML con corrección de bool→int

En pandas 2+, `get_dummies` devuelve columnas `bool`. Se fuerza conversión a `int` para evitar `ValueError` en sklearn.

In [ ]:
# Codificar sesión como variables dummy con dtype correcto
df_dummies = pd.get_dummies(df, columns=["session"], drop_first=False)

# Asegurar que las dummies sean int (corrección bug pandas 2+: get_dummies devuelve bool)
for col in ["session_asia", "session_europa", "session_america"]:
    if col not in df_dummies.columns:
        df_dummies[col] = 0
    else:
        df_dummies[col] = df_dummies[col].astype(int)  # CORRECCIÓN: bool → int

# Lista de features
feature_cols = [
    "ret_log", "vol_20", "vol_50",
    "ma_20", "ma_50", "signal_ma",
    "rsi_14", "bb_pos",
    "is_spike", "ticks_since_spike",
    "session_asia", "session_europa", "session_america",
]

# Filtrar filas sin NaN
data = df_dummies.dropna(subset=feature_cols + ["future_spike"]).copy()

X = data[feature_cols]
y = data["future_spike"]

print(f"Dataset: {len(X):,} muestras, {len(feature_cols)} features")
print(f"Distribución target:\n{y.value_counts()}")
print(f"Tasa positivos: {y.mean()*100:.2f}%")

# Verificar que no haya columnas bool
bool_cols = [c for c in X.columns if X[c].dtype == bool]
if bool_cols:
    raise ValueError(f"Columnas bool encontradas: {bool_cols}. Corregir antes de continuar.")
print("Tipos de datos correctos (sin bool).")
X.head()


## CELDA 9 — Modelo Random Forest con walk-forward validation

Correcciones respecto al notebook original:
- `max_depth=10` para evitar overfitting
- `class_weight='balanced'` para manejar desbalance de clases
- Split temporal (70/30) sin shuffle

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings("ignore")

# Walk-forward split (70% train, 30% test, sin shuffle para respetar orden temporal)
split_idx = int(len(X) * 0.70)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Train spikes: {y_train.sum()} | Test spikes: {y_test.sum()}")

# Verificar que haya ambas clases en train
if y_train.nunique() < 2:
    raise ValueError(f"y_train solo tiene una clase: {y_train.unique()}. Necesitas mas datos con spikes.")
if y_test.nunique() < 2:
    print("ADVERTENCIA: y_test solo tiene una clase. El reporte sera limitado.")

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,               # Limitar profundidad para evitar overfitting
    min_samples_leaf=20,        # Requiere mínimo 20 muestras por hoja
    class_weight="balanced",    # CORRECCIÓN: maneja desbalance de clases
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1] if 1 in rf.classes_ else np.zeros(len(X_test))

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

if y_test.nunique() >= 2:
    auc = roc_auc_score(y_test, y_proba)
    print(f"ROC-AUC: {auc:.4f}")

# Feature importance
fi = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\n--- Feature Importance ---")
print(fi.to_string())


## CELDA 10 — Estrategia y métricas con Sharpe correcto

Se corrige la división por cero en el cálculo del Sharpe ratio.

In [ ]:
# Crear signal_df alineado con X_test
signal_df = data.loc[X_test.index].copy()
signal_df["proba_spike"] = y_proba

# Regla de entrada
p_thr = 0.5
vol_min, vol_max = signal_df["vol_50"].quantile([0.1, 0.9])

signal_df["signal_model"] = 0
signal_df.loc[
    (signal_df["proba_spike"] >= p_thr) &
    (signal_df["ma_20"] > signal_df["ma_50"]) &
    (signal_df["vol_50"].between(vol_min, vol_max)),
    "signal_model"
] = 1

# Retorno logarítmico
signal_df["ret_log_clean"] = df.loc[signal_df.index, "ret_log"]

# Retorno estrategia (shift para evitar look-ahead)
signal_df["ret_strategy"] = signal_df["signal_model"].shift(1) * signal_df["ret_log_clean"]

# Curva de equity
signal_df["equity"] = signal_df["ret_strategy"].fillna(0).cumsum().apply(np.exp)

# Métricas
ret = signal_df["ret_strategy"].dropna()
n_trades = signal_df["signal_model"].sum()

mean_ret = ret.mean()
vol_ret  = ret.std()

# Sharpe anualizado correcto con manejo de división por cero
if vol_ret > 0:
    # Para ticks de 1 segundo: 3600*24*252 periodos/año
    sharpe = mean_ret / vol_ret * np.sqrt(3600 * 24 * 252)
else:
    sharpe = 0.0
    print("ADVERTENCIA: Volatilidad=0, no se generaron senales. Ajusta p_thr.")

max_dd = (signal_df["equity"].cummax() - signal_df["equity"]).max()
total_ret = signal_df["equity"].iloc[-1] - 1

# Sortino ratio
downside = ret[ret < 0].std()
sortino  = (mean_ret / downside * np.sqrt(3600 * 24 * 252)) if downside > 0 else 0.0

print(f"Senales generadas    : {n_trades:,}")
print(f"Retorno total        : {total_ret*100:.4f}%")
print(f"Retorno medio/tick   : {mean_ret:.8f}")
print(f"Volatilidad          : {vol_ret:.8f}")
print(f"Sharpe ratio         : {sharpe:.4f}")
print(f"Sortino ratio        : {sortino:.4f}")
print(f"Max drawdown         : {max_dd:.6f}")

signal_df[["proba_spike", "signal_model", "equity"]].tail()


## CELDA 11 — Guardar resultados en Drive

In [ ]:
# Guardar señales en Drive
output_path = os.path.join(CFG["SAVE_DIR"], "boom1000_rf_features_signals.parquet")
signal_df.to_parquet(output_path, compression="snappy")
print(f"Resultados guardados en: {output_path}")

# Guardar también CSV para compatibilidad
csv_path = os.path.join(CFG["SAVE_DIR"], "boom1000_rf_features_signals.csv")
signal_df.to_csv(csv_path)
print(f"CSV guardado en: {csv_path}")

print(f"\nResumen final:")
print(f"  Ticks recolectados : {len(df_master):,}")
print(f"  Features calculadas: {len(feature_cols)}")
print(f"  Muestras modelo    : {len(X):,}")
print(f"  Spikes detectados  : {df['is_spike'].sum():,}")
